# A&R Analyst Portfolio Project — Part 3
### Combined Scoring Model: Ranking Emerging Artists

**Prepared by:** Manav Desai

Building on Part 2's momentum signals, this section develops a weighted scoring formula to rank emerging artists by scouting potential — combining audience reach with fan engagement/loyalty.

## 1. Loading Data

In [10]:
# Load the emerging artist data generated in Part 2 (avoids re-running the API pipeline)
import pandas as pd

artist_df = pd.read_csv('data/emerging_artists.csv')
display(artist_df)

,artist,listeners,playcount,plays_per_listener
0,Ariana Grande,4615803,1184122911,256.5
1,Olivia Rodrigo,3286359,671116070,204.2
2,Steve Lacy,3237021,303323626,93.7
3,beabadoobee,2746077,345070556,125.7
4,Don Toliver,2644513,300399886,113.6
5,Noah Kahan,1681400,215010280,127.9
6,Malcolm Todd,1642759,134973953,82.2
7,Rainbow Kitten Surprise,1121895,45978020,41.0
8,mgk,1091323,43213080,39.6
9,Motionless In White,947626,65177335,68.8


In [11]:
# Confirm which columns carried over from Part 2's saved CSV
print(artist_df.columns.tolist())

['artist', 'listeners', 'playcount', 'plays_per_listener']


In [12]:
# The 'tier' column wasn't included in Part 2's saved CSV — recreating it here
# using the same logic, so this notebook works independently
def classify_artist(listeners):
    if listeners >= 1_000_000:
        return 'Established'
    elif listeners >= 100_000:
        return 'Emerging'
    else:
        return 'Early-Stage'

artist_df['tier'] = artist_df['listeners'].apply(classify_artist)
print(artist_df.columns.tolist())


['artist', 'listeners', 'playcount', 'plays_per_listener', 'tier']


## 2. Isolating the Actionable Scouting Pool

Filtering to the "Emerging" tier — artists with real audience traction but not yet at superstar saturation. This is the pool most relevant for scouting decisions.

In [13]:
# Filter to Emerging-tier artists — the actionable pool for scouting decisions
emerging_df = artist_df[artist_df['tier'] == 'Emerging']
print(emerging_df.shape)
display(emerging_df)

(7, 5)


,artist,listeners,playcount,plays_per_listener,tier
9,Motionless In White,947626,65177335,68.8,Emerging
10,Temper City,498427,3691874,7.4,Emerging
11,Ella Langley,483025,12948285,26.8,Emerging
12,Bella Kay,437120,11225131,25.7,Emerging
13,Dexter and The Moonrocks,330735,4782282,14.5,Emerging
14,Cameron Whitcomb,175771,4224122,24.0,Emerging
15,Tucker Wetmore,174846,3535098,20.2,Emerging


## 3. Normalizing Signals

Listener count and plays-per-listener are on very different scales. Applying min-max scaling brings both to a comparable 0-1 range before combining them into a single score.

In [8]:
# Normalize listeners to a 0-1 scale using min-max scaling
emerging_df['listeners_norm'] = (emerging_df['listeners'] - emerging_df['listeners'].min()) / (emerging_df['listeners'].max() - emerging_df['listeners'].min())

# Normalize plays_per_listener the same way
emerging_df['engagement_norm'] = (emerging_df['plays_per_listener'] - emerging_df['plays_per_listener'].min()) / (emerging_df['plays_per_listener'].max() - emerging_df['plays_per_listener'].min())

display(emerging_df)

,artist,listeners,playcount,plays_per_listener,tier,listeners_norm,engagement_norm
9,Motionless In White,947626,65177335,68.8,Emerging,1.000000,1.000000
10,Temper City,498427,3691874,7.4,Emerging,0.418723,0.000000
11,Ella Langley,483025,12948285,26.8,Emerging,0.398793,0.315961
12,Bella Kay,437120,11225131,25.7,Emerging,0.339390,0.298046
13,Dexter and The Moonrocks,330735,4782282,14.5,Emerging,0.201725,0.115635
14,Cameron Whitcomb,175771,4224122,24.0,Emerging,0.001197,0.270358
15,Tucker Wetmore,174846,3535098,20.2,Emerging,0.000000,0.208469


## 4. Building the Scouting Score

Combining reach (listener count) and loyalty (plays-per-listener) into a single weighted score, favoring engagement (60%) over raw reach (40%) — reflecting the view that a devoted niche fanbase is a stronger predictor of breakout potential than current audience size alone.

In [9]:
# Combine normalized signals into a single Scouting Score, weighting engagement slightly higher than reach
emerging_df['scouting_score'] = (0.4 * emerging_df['listeners_norm']) + (0.6 * emerging_df['engagement_norm'])

# Sort by score to get our final ranked shortlist
emerging_df = emerging_df.sort_values('scouting_score', ascending=False).reset_index(drop=True)
display(emerging_df)

,artist,listeners,playcount,plays_per_listener,tier,listeners_norm,engagement_norm,scouting_score
0,Motionless In White,947626,65177335,68.8,Emerging,1.000000,1.000000,1.000000
1,Ella Langley,483025,12948285,26.8,Emerging,0.398793,0.315961,0.349094
2,Bella Kay,437120,11225131,25.7,Emerging,0.339390,0.298046,0.314583
3,Temper City,498427,3691874,7.4,Emerging,0.418723,0.000000,0.167489
4,Cameron Whitcomb,175771,4224122,24.0,Emerging,0.001197,0.270358,0.162694
5,Dexter and The Moonrocks,330735,4782282,14.5,Emerging,0.201725,0.115635,0.150071
6,Tucker Wetmore,174846,3535098,20.2,Emerging,0.000000,0.208469,0.125081


**Takeaway:** Motionless In White ranks #1, excelling on both reach and engagement. Notably, Temper City — despite having the second-highest raw listener count in this pool — drops to #4 once engagement is factored in, since its audience shows minimal repeat listening. This demonstrates the scoring model surfaces genuinely different signal than popularity alone would suggest.

## Top Scouting Recommendations

1. **Motionless In White** — highest score on both reach and engagement; strongest overall candidate
2. **Ella Langley** — solid reach with above-average engagement
3. **Bella Kay** — comparable profile to Ella Langley, close third

## Methodology Notes & Limitations

- This scoring model reflects one reasonable weighting choice (60% engagement / 40% reach); different weightings would produce different rankings, and this tradeoff is a judgment call worth discussing openly
- Sample size is small (7 emerging artists across 4 genres) — a production version would benefit from a substantially larger artist pool
- Due to Spotify API restrictions on audio-feature data (documented in Part 2), this model relies solely on Last.fm momentum signals rather than combining them with the audio-characteristic findings from Part 1